# Step 6 (Kaggle) — DFT-PBE reference for HfO2 / ZrO2 polymorphs (Quantum ESPRESSO)
The MLIPs are trained on PBE (most models) or r2SCAN. To separate the error of the machine-learning model from the error of the DFT functional, we compute the same quantities with DFT ourselves.

**Run on Kaggle:** Settings (right panel) → Internet **On** (needs phone verification), Accelerator **None (CPU)**. Then click **Save Version → Save & Run All (Commit)**. The run continues on Kaggle's servers even if you close the browser (max 12 h). When it is finished, open the version → **Output** tab → download `dft_oxides_<MATERIAL>.csv`. Set MATERIAL in cell 3 ("HfO2" first, then "ZrO2" in a second notebook).


In [ ]:
# 1. Install Quantum ESPRESSO from conda-forge (3-5 min)
import os, subprocess
os.makedirs('/tmp/k', exist_ok=True)
if not os.path.exists('/tmp/k/qe/bin/pw.x'):
    !wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /tmp/k bin/micromamba
    !/tmp/k/bin/micromamba create -y -q -p /tmp/k/qe -c conda-forge qe > /tmp/k/mamba.log 2>&1; tail -3 /tmp/k/mamba.log
PW = '/tmp/k/qe/bin/pw.x'
assert os.path.exists(PW), 'Quantum ESPRESSO not installed - send the output above to Claude'
# make sure all shared libraries are found (fix soname mismatches, e.g. libelpa)
import glob, re
os.environ['LD_LIBRARY_PATH'] = '/tmp/k/qe/lib:' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['OMPI_ALLOW_RUN_AS_ROOT'] = '1'; os.environ['OMPI_ALLOW_RUN_AS_ROOT_CONFIRM'] = '1'
def missing_libs():
    out = subprocess.run(f'ldd {PW}', shell=True, capture_output=True, text=True).stdout
    return re.findall(r'(\S+) => not found', out)
for lib in missing_libs():
    stem = lib.split('.so')[0]
    cand = sorted(glob.glob(f'/tmp/k/qe/lib/{stem}*.so*'))
    print('missing', lib, '-> candidates', [os.path.basename(c) for c in cand])
    if cand: os.symlink(cand[-1], f'/tmp/k/qe/lib/{lib}')
if missing_libs():
    print('installing missing libraries:', missing_libs())
    subprocess.run('/tmp/k/bin/micromamba install -y -q -p /tmp/k/qe -c conda-forge elpa scalapack > /tmp/k/mamba2.log 2>&1', shell=True)
    for lib in missing_libs():
        cand = sorted(glob.glob(f"/tmp/k/qe/lib/{lib.split('.so')[0]}*.so*"))
        if cand: os.symlink(cand[-1], f'/tmp/k/qe/lib/{lib}')
print('still missing:', missing_libs())
assert not missing_libs(), 'Some libraries are still missing - send this output to Claude'
!pip install -q "ase>=3.23" spglib
MPI = None  # serial + OpenMP: MPI was about 4x slower on Kaggle
print('pw.x =', PW, '| mpirun =', MPI, '| CPUs =', os.cpu_count())


In [ ]:
# 2. Pseudopotentials (GBRV, fallback PseudoDojo)
import os, glob, subprocess, re
OUT = '/kaggle/working'  # only results here; os.makedirs(OUT, exist_ok=True)
SOURCES = {
 'gbrv': ('https://www.physics.rutgers.edu/gbrv/all_{xc}_UPF_v1.5.tar.gz', dict(ecutwfc=45, ecutrho=360)),
 'dojo': ('http://www.pseudo-dojo.org/pseudos/nc-sr-04_{xc}_standard_upf.tgz', dict(ecutwfc=85, ecutrho=340)),
}
PSEUDO = {}
def find_upf(d, el):
    pat = re.compile(rf'^{el}([_.-]|$)', re.I)
    return sorted(f for f in glob.glob(d + '/**/*', recursive=True)
                  if f.lower().endswith('.upf') and pat.match(os.path.basename(f)))
for xc in ['pbe']:
    for src, (url, cut) in SOURCES.items():
        d = f'/tmp/k/pseudo_{xc}_{src}'
        if not os.path.isdir(d) or not glob.glob(d + '/**/*.*', recursive=True):
            os.makedirs(d, exist_ok=True)
            r = subprocess.run(f'wget -q --tries=3 --timeout=60 "{url.format(xc=xc)}" -O /tmp/k/{xc}_{src}.tgz && tar xzf /tmp/k/{xc}_{src}.tgz -C {d}', shell=True)
        ok = all(find_upf(d, el) for el in ['Hf','Zr','O'])
        print(f'{xc:6s} {src}: download {"OK" if ok else "FAILED"}')
        if ok: PSEUDO[xc] = (d, cut, src); break
print('using:', {k: v[2] for k, v in PSEUDO.items()})
assert 'pbe' in PSEUDO, 'No PBE pseudopotentials could be downloaded - send this output to Claude'


In [ ]:
# 3. Structures and DFT relaxation
import numpy as np, pandas as pd, time, spglib
from ase import Atoms
from ase.io import write, read

XCS = ['pbe']
MATERIAL = "HfO2"        # second run: "ZrO2"
ORDER = ["Fm-3m", "P4_2/nmc", "P2_1/c", "Pca2_1", "Pbca"]   # cheapest first
NP = os.cpu_count()
XYZ = """12
Lattice="5.02792259 0.0 -0.81890993 0.0 5.1506447 0.0 -0.02837742 0.0 5.27613418" Properties=species:S:1:pos:R:3 material=HfO2 phase=P2_1/c pbc="T T T"
Hf       1.36830074       2.35932842       3.50888373
Hf       3.61705572       4.93465077       3.58640761
Hf       3.63124443       2.79131628       0.94834052
Hf       1.38248945       0.21599393       0.87081664
O        2.22472884       3.82448735       4.79623892
O        2.76062762       1.24916500       2.29905242
O        2.77481633       1.32615735      -0.33901467
O        2.23891755       3.90147970       2.15817183
O        0.33388695       1.70362147       1.77089481
O        4.67984693       4.27894382       0.04826235
O        4.66565822       3.44702323       2.68632944
O        0.31969824       0.87170088       4.40896190
12
Lattice="5.00476369 0.0 0.0 0.0 5.03268505 0.0 0.0 0.0 5.21946336" Properties=species:S:1:pos:R:3 material=HfO2 phase=Pca2_1 pbc="T T T"
Hf       1.33867009       1.21509324       0.17227361
Hf       1.33867009       3.73143576       2.43745807
Hf       3.66609360       1.21509324       2.78200529
Hf       3.66609360       3.73143576       5.04718975
O        0.35711291       1.90873267       1.92009730
O        0.35711291       4.42507519       0.68963438
O        2.30277660       2.47755255       3.83081741
O        2.30277660       4.99389508       3.99837763
O        2.70198709       4.99389508       1.38864595
O        2.70198709       2.47755255       1.22108573
O        4.64765078       1.90873267       4.52982898
O        4.64765078       4.42507519       3.29936606
24
Lattice="10.09373135 0.0 0.0 0.0 5.14134738 0.0 0.0 -0.0 5.25900202" Properties=species:S:1:pos:R:3 material=HfO2 phase=Pbca pbc="T T T"
Hf       1.39406704       4.93215542       1.80222281
Hf       1.39406704       2.77986565       4.43172382
Hf       3.65279864       2.36148173       1.80222281
Hf       3.65279864       0.20919196       4.43172382
Hf       6.44093271       4.93215542       0.82727820
Hf       6.44093271       2.77986565       3.45677921
Hf       8.69966431       2.36148173       0.82727820
Hf       8.69966431       0.20919196       3.45677921
O        0.33961712       4.27460403       3.50822440
O        0.33961712       3.43741704       0.87872339
O        2.25810407       1.27299525       0.46017377
O        2.25810407       1.29767844       3.08967478
O        2.78876161       3.84366894       0.46017377
O        2.78876161       3.86835213       3.08967478
O        4.70724856       0.86674335       0.87872339
O        4.70724856       1.70393034       3.50822440
O        5.38648279       3.43741704       1.75077762
O        5.38648279       4.27460403       4.38027863
O        7.30496974       1.27299525       2.16932724
O        7.30496974       1.29767844       4.79882825
O        7.83562728       3.84366894       2.16932724
O        7.83562728       3.86835213       4.79882825
O        9.75411423       0.86674335       1.75077762
O        9.75411423       1.70393034       4.38027863
6
Lattice="3.55801933 0.0 -0.0 0.0 3.55801933 0.0 0.0 0.0 5.17186121" Properties=species:S:1:pos:R:3 material=HfO2 phase=P4_2/nmc pbc="T T T"
Hf       1.77900967       1.77900967       2.58593061
Hf       0.00000000       0.00000000       0.00000000
O        0.00000000       1.77900967       3.60586239
O        1.77900967       0.00000000       4.15192943
O        1.77900967       0.00000000       1.56599882
O        0.00000000       1.77900967       1.01993178
3
Lattice="-2.51607961 -2.51607961 0.0 -2.51607961 0.0 -2.51607961 0.0 -2.51607961 -2.51607961" Properties=species:S:1:pos:R:3 material=HfO2 phase=Fm-3m pbc="T T T"
Hf       0.00000000       0.00000000       0.00000000
O       -1.25803980      -1.25803980      -1.25803980
O       -3.77411941      -3.77411941      -3.77411941
12
Lattice="5.08507902 0.0 -0.83997737 0.0 5.22810872 0.0 -0.00532399 0.0 5.3296935" Properties=species:S:1:pos:R:3 material=ZrO2 phase=P2_1/c pbc="T T T"
Zr       3.67581696       5.00776187       3.60471867
Zr       3.67847896       2.83440121       0.93987192
Zr       1.40127607       2.39370751       3.54984421
Zr       1.40393807       0.22034685       0.88499746
O        0.34743867       0.86937206       4.43921164
O        0.35010066       1.74468230       1.77436489
O        2.80022370       1.34582205      -0.34854063
O        2.28219333       3.95987641       2.17341001
O        2.79756170       1.26823231       2.31630612
O        2.27953133       3.88228667       4.83825676
O        4.72965437       3.48342642       2.71535124
O        4.73231636       4.35873666       0.05050449
12
Lattice="5.07466648 0.0 0.0 0.0 5.10110803 0.0 0.0 0.0 5.28340254" Properties=species:S:1:pos:R:3 material=ZrO2 phase=Pca2_1 pbc="T T T"
Zr       3.72245791       3.82101236       0.15995005
Zr       1.35220857       3.82101236       2.80165132
Zr       1.35220857       1.27045835       5.12345249
Zr       3.72245791       1.27045835       2.48175122
O        2.35108339       2.55481364       1.21453220
O        4.73283517       3.09183555       1.92860492
O        0.34183131       3.09183555       4.57030619
O        2.72358309       2.55481364       3.85623347
O        4.73283517       0.54128153       0.71309635
O        2.72358309       0.00425963       4.06887034
O        0.34183131       0.54128153       3.35479762
O        2.35108339       0.00425963       1.42716907
24
Lattice="10.21635077 0.0 0.0 0.0 5.21884202 -0.0 0.0 0.0 5.31038101" Properties=species:S:1:pos:R:3 material=ZrO2 phase=Pbca pbc="T T T"
Zr       1.41142523       2.82306981       4.46699230
Zr       1.41142523       5.00519322       1.81180179
Zr       3.69675016       2.39577221       1.81180179
Zr       3.69675016       0.21364880       4.46699230
Zr       6.51960061       5.00519322       0.84338871
Zr       6.51960061       2.82306981       3.49857922
Zr       8.80492554       2.39577221       0.84338871
Zr       8.80492554       0.21364880       3.49857922
O        0.34535321       4.35582508       3.55725935
O        0.34535321       3.47243795       0.90206885
O        2.29432952       1.29904446       0.46260784
O        2.29432952       1.31037655       3.11779835
O        2.81384587       3.90846547       0.46260784
O        2.81384587       3.91979756       3.11779835
O        4.76282217       0.86301694       0.90206885
O        4.76282217       1.74640407       3.55725935
O        5.45352860       3.47243795       1.75312166
O        5.45352860       4.35582508       4.40831216
O        7.40250490       1.29904446       2.19258266
O        7.40250490       1.31037655       4.84777317
O        7.92202125       3.90846547       2.19258266
O        7.92202125       3.91979756       4.84777317
O        9.87099756       0.86301694       1.75312166
O        9.87099756       1.74640407       4.40831216
6
Lattice="3.60016846 0.0 0.0 0.0 3.60016846 0.0 -0.0 0.0 5.22515604" Properties=species:S:1:pos:R:3 material=ZrO2 phase=P4_2/nmc pbc="T T T"
Zr       2.70012635       0.90004211       1.30628901
Zr       0.90004211       2.70012635       3.91886703
O        2.70012635       2.70012635       2.88311000
O        2.70012635       2.70012635       0.27053198
O        0.90004211       0.90004211       4.95462406
O        0.90004211       0.90004211       2.34204604
3
Lattice="3.11706474 0.0 1.7996377 1.03902225 2.93879676 1.7996377 -0.0 -0.0 3.59927642" Properties=species:S:1:pos:R:3 material=ZrO2 phase=Fm-3m pbc="T T T"
Zr       0.00000000       0.00000000       0.00000000
O        1.03902175       0.73469919       1.79963795
O        3.11706524       2.20409757       5.39891386
"""
import io
from ase.io import read as _read
STRUCT = {(a.info["material"], a.info["phase"]): a for a in _read(io.StringIO(XYZ), index=":", format="extxyz")}

def kgrid(at, density=20.0):
    return tuple(max(1, int(np.ceil(density / np.linalg.norm(v)))) for v in at.cell)

def spg(at, prec=1e-3):
    return spglib.get_spacegroup((at.cell[:], at.get_scaled_positions(), at.numbers), symprec=prec)

def pseudo(el, xc):
    f = find_upf(PSEUDO[xc][0], el)
    assert f, f'no pseudopotential for {el} {xc}'
    return f[-1]

def run_dft(at, xc, tag):
    wd = f'/tmp/k/run/{xc}/{tag}'; os.makedirs(wd, exist_ok=True)
    pp = {el: os.path.basename(pseudo(el, xc)) for el in set(at.get_chemical_symbols())}
    pdir = os.path.dirname(pseudo('O', xc))
    inp = dict(control=dict(calculation='vc-relax', prefix='x', pseudo_dir=pdir, outdir=wd,
                            etot_conv_thr=1e-5, forc_conv_thr=5e-4, nstep=200),
               system=dict(occupations='fixed', **PSEUDO[xc][1]),
               electrons=dict(conv_thr=1e-9, mixing_beta=0.5),
               ions=dict(ion_dynamics='bfgs'), cell=dict(cell_dynamics='bfgs', press_conv_thr=0.5))
    write(f'{wd}/pw.in', at, format='espresso-in', input_data=inp, pseudopotentials=pp, kpts=kgrid(at))
    t0 = time.time()
    cmd = (f'{MPI} -np {NP} --oversubscribe {PW} -nk {NP} -in pw.in' if MPI else f'OMP_NUM_THREADS={NP} {PW} -in pw.in')
    subprocess.run(f'cd {wd} && {cmd} > pw.out 2>&1', shell=True)
    if 'JOB DONE' not in open(f'{wd}/pw.out').read() and MPI:   # fall back to serial
        subprocess.run(f'cd {wd} && OMP_NUM_THREADS={NP} {PW} -in pw.in > pw.out 2>&1', shell=True)
    txt = open(f'{wd}/pw.out').read()
    fin = read(f'{wd}/pw.out', format='espresso-out', index=-1)
    return fin, ('End final coordinates' in txt or 'bfgs converged' in txt), time.time() - t0

csv = f'{OUT}/dft_oxides_{MATERIAL}.csv'
# resume: if a previous dft_results.csv was attached as input (Add Input -> Upload), start from it
prev = glob.glob(f'/kaggle/input/**/dft_oxides_{MATERIAL}*.csv', recursive=True)
if not os.path.exists(csv) and prev:
    import shutil; shutil.copy(prev[0], csv); print('resuming from', prev[0])
done = pd.read_csv(csv) if os.path.exists(csv) else pd.DataFrame()
for xc in XCS:
    for name in [MATERIAL]:
        for ph in ORDER:
            if len(done) and ((done.xc == xc) & (done.material == name) & (done.phase == ph)).any():
                continue
            start = STRUCT[(name, ph)]
            try:
                at, conv, dt = run_dft(start, xc, f"{name}_{ph.replace('/', '-')}")
            except Exception as e:
                print('FAILED', xc, name, ph, repr(e)[:300])
                po = f"/tmp/k/run/{xc}/{name}_{ph.replace('/', '-')}/pw.out"
                tail = ''.join(open(po).readlines()[-20:]) if os.path.exists(po) else 'no pw.out'
                raise RuntimeError('DFT run failed - send this message to Claude:\n' + repr(e)[:300] + '\n' + tail)
            cp = at.cell.cellpar()
            nfu = len(at) // 3
            row = dict(xc=xc, pseudo=PSEUDO[xc][2], material=name, phase=ph, nfu=nfu, E_eV_per_fu=at.get_potential_energy()/nfu,
                       a=cp[0], b=cp[1], c=cp[2], alpha=cp[3], beta=cp[4], gamma=cp[5], vol_per_fu=at.get_volume()/nfu,
                       kpts=str(kgrid(start)), spg_start=spg(start), spg_final=spg(at), converged=conv, time_s=round(dt))
            done = pd.concat([done, pd.DataFrame([row])], ignore_index=True); done.to_csv(csv, index=False)
            print(f"{xc} {name} {ph:9s} E/fu={row['E_eV_per_fu']:.4f} {row['spg_final']} conv={conv} {dt:.0f}s")


In [ ]:
# 4. Energies relative to the monoclinic phase
assert len(done), 'No DFT result yet - send the output of cell 3 to Claude'
E0 = done[done.phase == 'P2_1/c'].set_index('material').E_eV_per_fu
done['dE_meV_per_fu'] = 1000*(done.E_eV_per_fu - done.material.map(E0))
done.to_csv(csv, index=False)
print('Results in', csv, '- download it from the Output tab')
done[['material','phase','dE_meV_per_fu','vol_per_fu','spg_final','converged','time_s']].round(2)
